In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,split,explode,size
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = (
    SparkSession.builder
    .appName("iceberg-lab-hive-metastore")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.machine",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        "spark.sql.catalog.machine.type",
        "hive"
    )
    .config(
        "spark.sql.catalog.machine.uri",
        "thrift://hive-metastore:9083"       
    )
    .config(
        "spark.sql.catalog.machine.warehouse",
        "/data/warehouse"
    )
    .getOrCreate()
)

df = spark.read.parquet("/data/raw/machine-raw-log-anonymized.parquet").select("ChassisId","Cell","LogId","CellValue","CreationDateTime")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jmattam/.ivy2/cache
The jars for the packages stored in: /home/jmattam/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ca0e531e-c60b-48c3-a97c-1f5a73e8451c;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 in central
:: resolution report :: resolve 97ms :: artifacts dl 3ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.s

In [2]:
spark.sql("SHOW DATABASES IN machine").show()

+---------+
|namespace|
+---------+
|   bronze|
|  default|
|   silver|
+---------+



In [11]:
spark.sql("SHOW TABLES IN machine.bronze").show(truncate=False)

+---------+---------------+-----------+
|namespace|tableName      |isTemporary|
+---------+---------------+-----------+
|bronze   |machine_raw_log|false      |
+---------+---------------+-----------+



In [46]:
spark.sql("CREATE DATABASE IF NOT EXISTS machine.bronze")

DataFrame[]

In [9]:
df.show()

+-------------+----+-----+---------+--------------------+
|    ChassisId|Cell|LogId|CellValue|    CreationDateTime|
+-------------+----+-----+---------+--------------------+
|CHS66919BA506|   1| 1061|  2083007|2021-05-24 15:33:...|
|CHS66919BA506|   2| 1061|      577|2021-05-24 15:33:...|
|CHS66919BA506|   3| 1061|      598|2021-05-24 15:33:...|
|CHS66919BA506|   4| 1061|        0|2021-05-24 15:33:...|
|CHS66919BA506|   5| 1061|   536585|2021-05-24 15:33:...|
|CHS66919BA506|   6| 1061|        0|2021-05-24 15:33:...|
|CHS66919BA506|   7| 1061|      960|2021-05-24 15:33:...|
|CHS66919BA506|   1|  324|  2083005|2021-05-24 15:33:...|
|CHS66919BA506|   2|  324|   569190|2021-05-24 15:33:...|
|CHS66919BA506|   3|  324|   141905|2021-05-24 15:33:...|
|CHS66919BA506|   4|  324|   603270|2021-05-24 15:33:...|
|CHS66919BA506|   1|  420|  2083006|2021-05-24 15:33:...|
|CHS66919BA506|   2|  420|        0|2021-05-24 15:33:...|
|CHS66919BA506|   3|  420|        0|2021-05-24 15:33:...|
|CHS66919BA506

In [10]:
df.writeTo("machine.bronze.machine_raw_log").createOrReplace()

In [12]:
spark.sql("select * from  machine.bronze.machine_raw_log limit 5").show(truncate=False)

+-------------+----+-----+---------+-----------------------+
|ChassisId    |Cell|LogId|CellValue|CreationDateTime       |
+-------------+----+-----+---------+-----------------------+
|CHS66919BA506|1   |1061 |2083007  |2021-05-24 15:33:10.948|
|CHS66919BA506|2   |1061 |577      |2021-05-24 15:33:10.948|
|CHS66919BA506|3   |1061 |598      |2021-05-24 15:33:10.948|
|CHS66919BA506|4   |1061 |0        |2021-05-24 15:33:10.948|
|CHS66919BA506|5   |1061 |536585   |2021-05-24 15:33:10.948|
+-------------+----+-----+---------+-----------------------+



In [13]:
mapping_schema = StructType([
    StructField("PIPELINE", StringType(), True),
    StructField("LOG_ID", IntegerType(), True),
    StructField("Cell", StringType(), True),
    StructField("MEASURE_CODE_ACC", StringType(), True),
    StructField("UNIT", DoubleType(), True),
    StructField("TARGET_UNIT", StringType(), True),
    StructField("DESCRIPTION", StringType(), True)
])

df_mapping = spark.read.option("header", "true").schema(mapping_schema).csv("/data/raw/log_cell_mapping.csv")

In [14]:
df_mapping = df_mapping.withColumn("Cell_Array",split(df_mapping['Cell'],",")).withColumn("Cell_number",explode("Cell_Array")).withColumn("Cell_Count",size(col("Cell_Array")))

In [15]:
lkpDF = df_mapping.select("MEASURE_CODE_ACC","LOG_ID","Cell_number","UNIT","Cell_Array","Cell_Count","PIPELINE")
cond = [((df.LogId == lkpDF.LOG_ID) &\
         (df.Cell == lkpDF.Cell_number))]
mergedDF = df.join(lkpDF,on = cond, how = "inner" )
mergedDF = mergedDF.withColumn("CellValue_new",col("CellValue")*col("UNIT"))

In [16]:
groupbyCols = [
    "ChassisId",
    "CreationDateTime",
    "LogId",
    "MEASURE_CODE_ACC",
    "PIPELINE"
]
mergedDF = mergedDF.groupBy(groupbyCols).agg(F.sum("CellValue_new").alias("MEASURE_CODE_ACC_VALUE"))

In [17]:
mergedDF.show()

+-------------+--------------------+-----+----------------+--------+----------------------+
|    ChassisId|    CreationDateTime|LogId|MEASURE_CODE_ACC|PIPELINE|MEASURE_CODE_ACC_VALUE|
+-------------+--------------------+-----+----------------+--------+----------------------+
|CHS86CECE4B6D|2021-05-25 04:13:...| 1061| ENG_HOURS_TOTAL|  HAULER|    119097.15000000001|
|CHS86CECE4B6D|2021-05-30 23:43:...| 1061| TOTAL_FUEL_USED|  HAULER|              2812.403|
|CHS66919BA506|2021-05-29 22:33:...| 1061| ENG_HOURS_TOTAL|  HAULER|    123625.40000000001|
|CHS66919BA506|2021-05-30 02:03:...| 1061| ENG_HOURS_TOTAL|  HAULER|    124255.45000000001|
|CHS86CECE4B6D|2021-05-29 15:13:...| 1061| ENG_HOURS_TOTAL|  HAULER|             137831.45|
|CHS86CECE4B6D|2021-05-29 17:13:...| 1061| TOTAL_FUEL_USED|  HAULER|              2764.791|
|CHS86CECE4B6D|2021-05-27 17:13:...| 1061| TOTAL_FUEL_USED|  HAULER|    2592.0510000000004|
|CHS86CECE4B6D|2021-05-28 02:13:...| 1061| TOTAL_FUEL_USED|  HAULER|            

In [18]:
spark.sql("CREATE DATABASE IF NOT EXISTS machine.silver")

DataFrame[]

In [19]:
mergedDF.writeTo("machine.silver.agg_raw_log").createOrReplace()

In [22]:
spark.sql("SHOW TABLES IN machine.silver").show()

+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|   silver|agg_raw_log|      false|
+---------+-----------+-----------+



In [3]:
spark.sql("select * from machine.silver.agg_raw_log limit 5").show()

[Stage 1:>                                                          (0 + 1) / 1]

+-------------+--------------------+-----+----------------+--------+----------------------+
|    ChassisId|    CreationDateTime|LogId|MEASURE_CODE_ACC|PIPELINE|MEASURE_CODE_ACC_VALUE|
+-------------+--------------------+-----+----------------+--------+----------------------+
|CHS86CECE4B6D|2021-05-25 04:13:...| 1061| ENG_HOURS_TOTAL|  HAULER|    119097.15000000001|
|CHS86CECE4B6D|2021-05-30 23:43:...| 1061| TOTAL_FUEL_USED|  HAULER|              2812.403|
|CHS66919BA506|2021-05-29 22:33:...| 1061| ENG_HOURS_TOTAL|  HAULER|    123625.40000000001|
|CHS66919BA506|2021-05-30 02:03:...| 1061| ENG_HOURS_TOTAL|  HAULER|    124255.45000000001|
|CHS86CECE4B6D|2021-05-29 15:13:...| 1061| ENG_HOURS_TOTAL|  HAULER|             137831.45|
+-------------+--------------------+-----+----------------+--------+----------------------+



In [4]:
spark.sql("update machine.silver.agg_raw_log set measure_code_acc_value = round(measure_code_acc_value,2)")

DataFrame[]